# A simple plan to investigate spin-glass behavior

**First, check that Monte Carlo reproduces the exact overlap distribution on a small lattice. Then study larger lattices to investigate the phase.**

Start with **2 × 2 Kagome: 12 spins, periodic boundaries, Gamma = 0.5 and h = 0.15**. Use the same random antiferromagnetic bonds as in our [existing overlap notebook](exact_guide_overlap_distributions.ipynb).

**First milestone executed on AutoDL, 30 October 2026:** see the [results notebook](overlap_disorder_benchmark.ipynb). It compares the same eight disorders across seven population/time settings, with 224 population records (220 new runs and four reused runs). At the strongest checked setting, Q₂ is 0.47148 from GFMC versus 0.47088 exactly. Remaining Monte Carlo uncertainty and ancestry warnings are reported there; larger-size phase identification remains pending.

## What are we measuring?

Take two independent spin snapshots of the same system. Compare the spins at each site: agreement scores +1, disagreement scores −1. Their average is the overlap:

$$q=\frac{\text{number of agreements}-\text{number of disagreements}}{12}.$$

**P(q)** tells us how likely each overlap value is. A peak at positive q means that the two snapshots often resemble each other.

We also calculate the finite-system quantity used in our notebook:

$$q_{\mathrm{EA}}^{\mathrm{finite}}=\frac1{12}\sum_{i=1}^{12}\langle\sigma_i^z\rangle^2=\langle q\rangle.$$

The brackets mean an average over measurements for one fixed set of bonds. In words: average each site's spin, square that average, then average over sites. Equivalently, take the mean of P(q). Its mean and its peak can be different.

**A positive overlap alone does not prove a spin glass.** An applied field can also make independent snapshots resemble each other. This finite-system quantity is useful for comparing exact and Monte Carlo results; identifying glass order requires the larger-size test below.

## First experiment: exact versus Monte Carlo

A **disorder sample** means one random set of bond strengths. Keep those strengths fixed while measuring it.

1. **Choose eight disorder samples.** Use the existing samples 50000–50007. Exact and Monte Carlo calculations must use the same eight samples.
2. **Calculate the exact answer.** For each sample, use the saved exact ground state to obtain P(q), its mean overlap, and its mean squared overlap using the formula below.
3. **Calculate the Monte Carlo answer.** Run guided GFMC four independent times for each sample and measure the same quantities. Average the four runs within that sample.
4. **Average over disorder.** Average the eight exact distributions to get one exact curve. Average the eight Monte Carlo distributions to get one Monte Carlo curve. Give every disorder equal weight.
5. **Compare the curves.** Show error bars from variation across the eight disorder samples. Also check agreement for individual samples. Increase the walker population and projection time to check that the Monte Carlo answer is stable and approaches the exact result.

**The output is one plot and one small table:**

- Plot: disorder-averaged exact and Monte Carlo P(q), with their mean overlaps marked.
- Table: disorder-averaged mean overlap and mean squared overlap Q₂ from each method, with error bars.

The matching eight-sample exact/GFMC coverage is now complete; see the linked results notebook. The original one-disorder calculation was extended with population and time checks. Agreement and remaining uncertainty are assessed together before calling the settings converged.

**Compute platform: AutoDL.** Run this benchmark and later large simulation campaigns on AutoDL. For large campaigns, measure GPU batching and concurrent CPU/GPU worker throughput, then use the combination that completes the most reliable measurements per hour within the available resources. Keep results resumable and mirror them locally for plotting and review. The 12-spin benchmark uses a lookup of the frozen neural guide; larger systems require direct neural inference.

## Computing the squared overlap exactly

**No Monte Carlo sampling of spin configurations is needed for the exact 12-spin calculation.** Exact diagonalization gives the ground-state amplitude for all 4096 spin configurations. For a normalized ground state, the probability of configuration s is its squared amplitude.

For one fixed disorder sample J, calculate each pair correlation by a complete sum:

$$\langle\sigma_i^z\sigma_j^z\rangle_J=\sum_s|\psi_J(s)|^2s_i s_j.$$

Here sᵢ is +1 or −1, and the sum includes all 4096 configurations. Two independent replicas with the same bonds then give

$$\langle q^2\rangle_J=\frac1{N^2}\sum_{i,j}\langle\sigma_i^z\sigma_j^z\rangle_J^2=\sum_q q^2P_J(q),\qquad N=12.$$

The first expression says: square each pair correlation, sum over all ordered site pairs including i = j, and divide by the number of spins squared. The second gives the same answer directly from P(q): multiply each bin's probability by q² and sum over the 13 bins. Independence of the two replicas is what makes the pair correlation appear squared.

**Why does the correlation get squared?** Keep one disorder sample fixed. Let a and b be independent snapshots drawn from the same ground-state distribution.

1. Define the overlap: $q=N^{-1}\sum_i s_i^{(a)}s_i^{(b)}$.
2. Square and group by snapshot: $q^2=N^{-2}\sum_{i,j}(s_i^{(a)}s_j^{(a)})(s_i^{(b)}s_j^{(b)})$.
3. Average: $\langle q^2\rangle_J=N^{-2}\sum_{i,j}\langle s_i^{(a)}s_j^{(a)}\rangle_J\langle s_i^{(b)}s_j^{(b)}\rangle_J=N^{-2}\sum_{i,j}\langle\sigma_i^z\sigma_j^z\rangle_J^2$.

The average factorizes because the snapshots are independent, and the two factors are equal because they have the same distribution; spins within each snapshot can still be correlated.

Repeat this calculation for all eight disorders and average their results equally to obtain **Q₂ at the chosen Gamma and h**. GFMC estimates the same quantity by sampling instead of summing over all configurations.

There are two different sources of sampling: the exact calculation eliminates spin-configuration sampling, but we still use only eight randomly drawn bond realizations. Thus each exact finite-system answer is accurate up to numerical precision, while its disorder average still has uncertainty from the limited number of disorders.

## How do we then test for a spin-glass phase?

**Study what happens as the lattice grows.** Begin this test at h = 0, where the field cannot create a positive overlap. Compare low and high transverse fields, initially Gamma = 0.5 and 2, without assuming which phase either point belongs to.

At zero field, positive and negative overlaps can cancel in the mean. To avoid losing that information, measure the squared overlap:

$$Q_2=\text{disorder average of }\langle q^2\rangle.$$

This measures how far the overlap lies from zero, regardless of its sign.

Use 2 × 2 as the exact benchmark, then obtain reliable Monte Carlo results for 4 × 4, 6 × 6 and larger sizes as feasible. Average over disorder at every size and check the Monte Carlo accuracy again as size increases.

| As the system grows… | What it suggests |
|---|---|
| P(q) narrows toward zero and Q₂ tends to zero | A paramagnet |
| Weight remains away from zero and Q₂ approaches a nonzero value | Persistent overlap order |

**We must also check whether the spins form an ordinary magnetic pattern**, such as aligned spins or a repeating antiferromagnetic pattern. Such patterns can produce overlap order too. Persistent overlap without ordinary magnetic order supports a spin-glass interpretation. Several sizes are needed to distinguish these possibilities.

## What about our original point, h = 0.15?

The first exact-versus-Monte-Carlo comparison still uses Gamma = 0.5 and h = 0.15. It tells us whether we can measure P(q) accurately there.

To investigate the phase at nonzero field, measure **connected correlations**. For one fixed disorder sample,

$$C_{ij}=\langle\sigma_i^z\sigma_j^z\rangle-\langle\sigma_i^z\rangle\langle\sigma_j^z\rangle.$$

The first term measures how two spins vary together. The second subtracts the contribution from their average orientations. For example, two independent spins can both prefer pointing up, but their connected correlation is zero. A nonzero Cᵢⱼ means their fluctuations are correlated.

To combine this information across the lattice and disorder samples, use the **equal-time connected spin-glass susceptibility**:

$$\chi_{\mathrm{conn}}=\frac1N\sum_{i,j}[C_{ij}^{2}]_D.$$

Here N is the number of spins and the square brackets mean an average over disorder samples. Square each correlation before averaging so that positive and negative correlations do not cancel. The sum includes i = j, so even independent fluctuating spins contribute a local background.

Compute this quantity from exact diagonalization and Monte Carlo for the same disorders, then examine how it changes with lattice size. Growth can indicate increasingly collective correlations, but a large value at one size does not establish a spin-glass phase. This is an equal-time correlation measure; it is not an imaginary-time-integrated response.

**Our immediate goal is the eight-disorder comparison on 2 × 2 Kagome. The phase question comes after the measurement workflow works.**

For simulation settings and later scaling details, see the [production plan](guided_gfmc_production_plan.ipynb).